# Install Dependencies

In [ ]:
!pip install -q --no-deps xformers trl peft accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.5/31.5 MB 59.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.3/366.3 kB 31.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.0/67.0 MB 38.1 MB/s eta 0:00:00


In [ ]:
!pip install -q datasets
!pip install -q regex

In [ ]:
!pip install -q emoji
!pip install -q PyArabic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 590.6/590.6 kB 32.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.4/126.4 kB 9.0 MB/s eta 0:00:00


In [ ]:
!pip install -q diffusers

# login

In [ ]:
import huggingface_hub
huggingface_hub.login('HF_TOKEN')

# Import Required Modules

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ['CUDA_LAUNCH_BLOCKING']="1"
os.environ['TORCH_USE_CUDA_DSA'] = "1"

In [ ]:
import warnings
warnings.filterwarnings("ignore")

In [ ]:
import numpy as np
import pandas as pd
import random
from sklearn.utils import shuffle
import os
import re
from tqdm import tqdm
import bitsandbytes as bnb
import torch
import torch.nn as nn
import transformers
from datasets import Dataset
from peft import LoraConfig, PeftConfig
from trl import SFTTrainer
from transformers import (AutoModelForCausalLM,
                          AutoTokenizer,
                          BitsAndBytesConfig,
                          TrainingArguments,
                          pipeline,
                          logging)
from sklearn.metrics import (accuracy_score,
                             classification_report,
                             precision_score,
                             recall_score,
                             f1_score,
                             confusion_matrix)
from sklearn.model_selection import train_test_split
import emoji
import pyarabic.araby as araby

In [ ]:
import pandas as pd

In [ ]:
import torch
import torch.distributed as dist

# Load Model

In [ ]:
model_name = "Qwen/Qwen2.5-7B-Instruct"

compute_dtype = getattr(torch, "float16")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=False,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    # quantization_config=bnb_config,
    device_map={"": 0},
    trust_remote_code=True,
)

model.config.use_cache = False
model.config.pretraining_tp = 1

tokenizer = AutoTokenizer.from_pretrained(model_name,
                                          trust_remote_code=True,
                                          padding_side="left",
                                          add_eos_token=True,
                                         )

# Assign pad_token if missing
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "left"

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/27.8k [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/3.95G [00:00<?, ?B/s]

model-00004-of-00004.safetensors:   0%|          | 0.00/3.56G [00:00<?, ?B/s]

...

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

In [ ]:
# pipe = pipeline(task="text-generation",
#                 model=model,
#                 tokenizer=tokenizer,
#                 max_new_tokens=20,
#                 temperature=0.2
#                )

Device set to use cuda:0


# Load Data

In [ ]:
import pandas as pd
data = pd.read_excel('sampled_data.xlsx')

In [ ]:
data.head()

,main directory,subdirectory,content
0,Khaleej,Culture,استقبل الوسط المسرحي الإماراتي فوز الإمارات بر...
1,Khaleej,Culture,استضاف مركز الشارقة لفن الخط العربي والزخرفة م...
2,Khaleej,Culture,باسمة يونس قد يبدو العنوان اسماً لرواية؛ لكنه ...
3,Khaleej,Culture,أبوظبي: «الخليج» أكد عدد من الخبراء والمسؤولين...
4,Khaleej,Culture,يمكن القول باطمئنان أن شهر رمضان المبارك هو شه...


# Zero Shot

## Predict Arabic Prompt

In [ ]:
content = '''أنت نموذج لغوي مكلف بتصنيف مقالات الصحف العربية إلى واحدة من الفئات التحريرية المحددة مسبقًا، وذلك بالاعتماد فقط على الموضوع الرئيسي ومحتوى المقال. اقرأ كل مقال بعناية، ثم حدد فئة واحدة فقط من الفئات التالية تعكس بشكل أفضل الموضوع الأساسي للمقال:
1. الثقافة
2. المال
3. الطب
4. السياسة
5. الدين
6. الرياضة
7. التكنولوجيا
'''

prompt = f'''المقال:
{data['content'].iloc[0]}
الفئة المتوقعة:'''

messages = [
    {"role": "system", "content": content},
    {"role": "user", "content": prompt}
]
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

generated_ids = model.generate(
    **model_inputs,
    max_new_tokens=512
)
generated_ids = [
    output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
]

response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
response

'الفئة المتوقعة هي: الثقافة\n\nهذا المقال يركز بشكل أساسي على فوز الإمارات برئاسة الهيئة الدولية للمسرح وتأثير هذا الفوز على الساحة المسرحية الإماراتية والعالمية. المقال يتناول الجهود والإنجازات المتعلقة بالمسرح في الإمارات، مما يجعل فئة "الثقافة" هي الأنسب لتصنيف هذا المقال.'

In [ ]:
content = '''أنت نموذج لغوي مكلف بتصنيف مقالات الصحف العربية إلى واحدة من الفئات التحريرية المحددة مسبقًا، وذلك بالاعتماد فقط على الموضوع الرئيسي ومحتوى المقال. اقرأ كل مقال بعناية، ثم حدد فئة واحدة فقط من الفئات التالية تعكس بشكل أفضل الموضوع الأساسي للمقال:
1. الثقافة
2. المال
3. الطب
4. السياسة
5. الدين
6. الرياضة
7. التكنولوجيا
'''
zero_pred = []
for text in data['content']:
    prompt = f'''المقال:
    {text}
    الفئة المتوقعة:'''
    messages = [
        {"role": "system", "content": content},
        {"role": "user", "content": prompt}
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=512
    )
    generated_ids = [
        output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]

    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    zero_pred.append(response)

In [ ]:
pred_zero = pd.DataFrame()
pred_zero['Predicted'] = zero_pred
pred_zero['Predicted'].value_counts()

,count
Predicted,
السياسة,151
الرياضة,116
التكنولوجيا,83
الفئة المتوقعة: المال,83
الفئة المتوقعة: الثقافة,70
...,...
الفئة المتوقعة: الرياضة\n\nالتحليل: رغم أن المقال يتحدث عن مبيعات هواتف ذكية وتلفزيونات ذكية، إلا أن المحتوى الرئيسي يركز على الأرقام القياسية للمبيعات التي حققها هذان الجهازان في دقائق قليلة بعد الإطلاق. هذه النوعية من الأخبار عادة ما تُصنف تحت قسم الرياضة في معظم الصحف، حيث يتم التركيز على الإنجازات والنتائج السريعة والرائعة في مجال الأعمال والتكنولوجيا.,1
"الفئة المتوقعة: المال\n\nالسبب: المقال يتناول حظر استيراد أجهزة بسبب انتهاك براءات الاختراع، وهو موضوع يتعلق بشكل مباشر بالقانون التجاري والاقتصادي، مما يجعل فئة ""المال"" هي الأنسب من بين الخيارات المعطاة.",1
ال tecnologia,1


In [ ]:
nor_pre = []
for pr in pred_zero['Predicted']:
  if (
      "الرياضة" in pr
      or "رياضة" in pr
      ):
    nor_pre.append("Sports")
  elif "الصحة" in pr:
    nor_pre.append("Medical")
  elif "الطب" in pr:
    nor_pre.append("Medical")
  elif "الثقافة" in pr:
    nor_pre.append("Culture")
  elif "المال" in pr:
    nor_pre.append("Finance")
  elif "السياسة" in pr:
    nor_pre.append("Politics")
  elif "الدين" in pr:
    nor_pre.append("Religion")
  elif (
      "التكنولوجيا" in pr
      or "technologia" in pr
      or "tecnología" in pr
      ):
    nor_pre.append("Tech")
  else:
    nor_pre.append("Unclassified")

pred_zero['Normalized Category'] = nor_pre

In [ ]:
pred_zero['Normalized Category'].value_counts()

,count
Normalized Category,
Politics,196
Culture,166
Sports,155
Finance,144
Medical,141
Tech,121
Religion,76
Unclassified,1


In [ ]:
pred_zero['Article'] = data['content']
pred_zero.to_excel('Qwen-NC-ZeroShot-ArabicPrompt.xlsx', index = False)

In [ ]:
y_true = data['subdirectory'].values
print(classification_report(y_true, pred_zero['Normalized Category'].values, digits = 4))

              precision    recall  f1-score   support

     Culture     0.7590    0.8400    0.7975       150
     Finance     0.7708    0.7400    0.7551       150
     Medical     0.9362    0.8800    0.9072       150
    Politics     0.7245    0.9467    0.8208       150
    Religion     0.9737    0.7400    0.8409       100
      Sports     0.9355    0.9667    0.9508       150
        Tech     0.9421    0.7600    0.8413       150
Unclassified     0.0000    0.0000    0.0000         0

    accuracy                         0.8440      1000
   macro avg     0.7552    0.7342    0.7392      1000
weighted avg     0.8576    0.8440    0.8450      1000



## Predict English Prompt

In [ ]:
content = '''You are a language model tasked with classifying Arabic newspaper articles into one of the predefined editorial categories based solely on the article's main topic and content. Carefully read each article and assign only one of the following categories that best reflects its primary subject matter:
1. Culture
2. Finance
3. Medical
4. Politics
5. Religion
6. Sports
7. Tech
'''
zero_pred = []
for text in data['content']:
    prompt = f'''Article:
    {text}
    Predicted Category:'''
    messages = [
        {"role": "system", "content": content},
        {"role": "user", "content": prompt}
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=512
    )
    generated_ids = [
        output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]

    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    zero_pred.append(response)

In [ ]:
pred_zero = pd.DataFrame()
pred_zero['Predicted'] = zero_pred
pred_zero['Predicted'].value_counts()

,count
Predicted,
Predicted Category: Medical,152
Politics,104
Sports,102
Culture,96
Predicted Category: Tech,86
...,...
"Predicted Category: Politics\n\nThe article primarily discusses the potential coaching prospects for the Moroccan national football team, which falls under political and sports-related news in the context of national and international football management and strategy discussions.",1
"Predicted Category: Politics\n\nThis article primarily discusses the potential hiring of a foreign coach to manage the Moroccan national football team, which is more related to sports management and national politics in sports rather than just sports itself. Therefore, it fits best under the ""Politics"" category.",1
"Predicted Category: Politics\n\nNote: While this article discusses a sports figure (Tata Martino, the coach of Barcelona), it focuses more on his dissatisfaction with the political and managerial situation at the club rather than sports performance or achievements. Therefore, it can be categorized under Politics due to its emphasis on the internal dynamics and decision-making processes within the football club.",1


In [ ]:
nor_pre = []
for pr in pred_zero['Predicted']:
  if (
      "الرياضة" in pr
      or "Sports" in pr
      or "sports" in pr
      ):
    nor_pre.append("Sports")
  elif (
      "الصحة" in pr
      or "الطب" in pr
      or "medicine" in pr
      or "Medicine" in pr
      or "Medical" in pr
      or "medical" in pr
      ):
    nor_pre.append("Medical")
  elif (
      "الثقافة" in pr
      or "Culture" in pr
      or "culture" in pr
      ):
    nor_pre.append("Culture")
  elif (
      "المال" in pr
      or "Finance" in pr
      or "finance" in pr
      ):
    nor_pre.append("Finance")
  elif (
      "السياسة" in pr
      or "politics" in pr
      or "Politics" in pr
      or "POLITICS" in pr
      ):
    nor_pre.append("Politics")
  elif (
      "الدين" in pr
      or "religion" in pr
      or "Religion" in pr
      or "religious" in pr
      or "Religious" in pr
      ):
    nor_pre.append("Religion")
  elif (
      "التكنولوجيا" in pr
      or "Tech" in pr
      or "tech" in pr
      or "Technology" in pr
      or "technology" in pr
      ):
    nor_pre.append("Tech")
  else:
    nor_pre.append("Unclassified")

pred_zero['Normalized Category'] = nor_pre

In [ ]:
pred_zero['Article'] = data['content']
pred_zero.to_excel('Qwen-NC-ZeroShot-EnglishPrompt.xlsx', index = False)

In [ ]:
y_true = data['subdirectory'].values
print(classification_report(y_true, pred_zero['Normalized Category'].values, digits = 4))

              precision    recall  f1-score   support

     Culture     0.8082    0.7867    0.7973       150
     Finance     0.7929    0.7400    0.7655       150
     Medical     0.8944    0.9600    0.9260       150
    Politics     0.6715    0.9267    0.7787       150
    Religion     0.9375    0.7500    0.8333       100
      Sports     0.9384    0.9133    0.9257       150
        Tech     0.9583    0.7667    0.8519       150

    accuracy                         0.8390      1000
   macro avg     0.8573    0.8348    0.8398      1000
weighted avg     0.8533    0.8390    0.8401      1000



# Pred Few Shot

In [ ]:
content = '''You are a language model tasked with classifying Arabic newspaper articles into one of the predefined editorial categories based solely on the article's main topic and content. Carefully read each article and assign only one of the following categories that best reflects its primary subject matter:
1. Culture
2. Finance
3. Medical
4. Politics
5. Religion
6. Sports
7. Tech

Example 1
Article content:
أكد الفنان عزت أبوعوف، رئيس مهرجان القاهرة السينمائي، إلغاء حفل ختام المهرجان الذي كان من المزمع أن يقام غداً، وذلك بسبب الظروف غير المستقرة التي تمر بها مصر حالياً.ونفى أبوعوف في تصريحات لـ"العربية.نت" سفر الضيوف الأجانب خوفاً مما تشهده مصر من مظاهرات، وما تردد عن أن هذا السفر قد تسبب في عدم إمكانية إقامة حفل الختام، مؤكداً أن وزير الثقافة دكتور محمد صابر عرب هو من أصدر قرار الإلغاء.وعن توزيع الجوائز التي كان من المزمع أن يتم في حفل الختام، شرح أنه سيقام مؤتمر صحافي سيكون بديلاً عن الحفل وسيتم من خلاله إعلان توزيع الجوائز والأفلام الفائزة في هذه الدورة.يُذكر أن حفل افتتاح المهرجان أيضاً تم في هدوء وبعيداً عن الصخب المعتاد، وذلك أيضاً بسبب الظروف نفسها التي تشهدها مصر.

Predicted Category: Culture

Example 2
Article content:
قال الرئيس التنفيذي للشركة السعودية للكهرباء زياد الشيحة، في مقابلة عبر الهاتف مع قناة "العربية"، إنه لأول مرة في تاريخ الشركة تراجع استهلاك المملكة في فترات الذروة. وأضاف الشيحة أن التراجع الذي حدث في استهلاك المملكة في 2016، مقارنة مع عام 2015، دفع الشركة لمراجعة السعات المطلوبة لدى دراسة المشاريع الجديدة. وأكد أن مسألة انخفاض الحمل الذروي لأول مرة في تاريخ الشركة عن العام جلعنا نراجع المحطات المستقبلية والتي ستكون بعقود شراء الطاقة، وهذا سيكون لمشاربع الإنتاج وتتم مراجعة السعات المطلوبة خاصة مع قلة الحمل الذروي في 2016. وستزود الشركة السعودية للكهرباء شركة زين السعودية بشبكة الألياف البصرية الممتدة لـ 60 ألف كم. وأوضح الشيحة أن "قطاع التوليد سيطرح للخصخصة كما هو معلن، ونعمل على الموضوع بشكل متوازن وشبه يومي". وكانت خسائر شركة السعودية للكهرباء قد تفاقمت بأكثر من 60%، في الربع الأخير من العام الماضي، مقارنة بالربع المماثل من عام 2015، لتبلغ 2.34 مليار ريال. من ناحية أخرى، ارتفعت أرباح الشركة بنسبة 37%، خلال العام الماضي، مقارنةً بعام 2015، لتبلغ 2.1 مليار ريال. وأرجعت الشركة تفاقم الخسائر الفصلية إلى ارتفاع تكلفة المبيعات نتيجة الزيادة في أسعار الوقود وارتفاع المصاريف التشغيلية.

Predicted Category: Finance

Example 3
Article content:
يعتقد بعض المدخنين أن السجائر الإلكترونية تعد أحد أهم العوامل المساعدة في الإقلاع عن التدخين، في حين يعتقد البعض الآخر أنها تعتبر وسيلة إغواء للاستمرار في الخضوع لتلك العادة المدمرة، إلا أن الأبحاث الطبية الحديثة تشير إلى أن الأخطار والفوائد لتلك النوعية من السجائر لاتزال غير معلومة بصورة واضحة بين المدخنين، وذلك وفق ما نشرت وكالة أنباء الشرق الأوسط المصرية. وكانت مجموعة من الباحثين قد أجرت أبحاثها على أكثر من 64 مدخنا، ولم ينجحوا في تحقيق إجماع حول الفوائد والأضرار المحتملة للسجائر الإلكترونية، وهو ما قد يعكس انقساما في المجتمع الطبي حول مدى ملاءمة تعزيز السجائر الإلكترونية كبديل أكثر أمنا للتدخين. وأوضح الباحثون أن معظم المشاركين في الدراسة يرون أن التدخين يعتبر شكلا من الإدمان، حيث تلعب الإرادة دورا قويا في الإقلاع عن هذه العادة المدمرة، في الوقت الذى حاول فيه جميع المشاركين في الدراسة مرة واحدة على الأقل الإقلاع عن العادة المدمرة.

Predicted Category: Medical

Example 4
Article content:
أعرب المتحدث باسم الهيئة العليا للمفاوضات السورية سالم المسلط عن أمله في أن تنتقل روسيا فعليا لتقف إلى جانب الشعب السوري بدلا من النظام، وذلك عقب قراره بسحب القوات الروسية من سوريا. وأضاف المسلط أن هناك جدية لمست مؤخرا حيال المواقف الروسية للدفع نحو الحل السياسي للأزمة في سوريا، خلال جولة المحادثات الجديدة التي انطلقت في جنيف. هذا وأعلن متحدث باسم الرئيس الروسي فلاديمير بوتين بوتين بأن روسيا أبلغت الأسد بقرار سحب الجزء الرئيسي من القوات الروسية من سوريا. وقال المتحدث إن بوتين خلال اجتماعه بوزير دفاعه أمر اعتبارا من اليوم (الثلاثاء) ببدء سحب الجزء الرئيسي من القوات الروسية. في حين قال متحدث باسم بوتين إن القاعدة البحرية والجوية الروسية في سوريا تستمر في العمل كما في السابق. ووفقا للكرملين فإن بوتين طلب من وزير خارجيته سيرغي لافروف تكثيف الدور الروسي في عملية السلام في سوريا، مشيرا إلى ان  القوات الروسية في سوريا أوجدت ظروفا ملائمة لعملية السلام.

Predicted Category: Politics

Example 5
Article content:
أكد عبداللطيف بخاري، رئيس لجنة الخبراء باتحاد القدم السعودي أن اتحاده يبحث عن 15 منصباً في اللجان الآسيوية، وأن هناك لجنة برئاسة خالد المرزوقي، عضو مجلس إدارة الاتحاد مكلفة بالاختيار. وقال بخاري لـ"في المرمى":" الترشيحات تتم بناء على معايير قارية، ونحن نبحث عن 15 منصباً في لجان الاتحاد الآسيوي". وبين بخاري أن لجنة المسابقات اقترحت إيجاد مراقب لكل مباراة، وزاد:" تقارير المراقب لن تغني عن تقارير الحكم، أما بالنسبة لتقارير الأول فيمكن للجنة الانضباط الاستناد عليها".

Predicted Category: Sports

Example 6
Article content:
يبدو أن هاتف #آيفون7  الجديد الذي ستصدره شركة آبل، لن يحمل الكثير من التغيرات، بحسب ما أفاد تقرير لـ "وول ستريت جورنول". فالهاتف الجديد سيأتي شبيها بالنسخة الحالية (آيفون6)، مع تغيير جذري على صعيد "الصوت" والسماعات. وحسب تقرير الصحيفة قد تزيل شركة #آبل منفذ سماعة الصوت، لتدمجه بالمنفذ الذي يوضع فيه شاحن الهاتف في الأسفل. وتراهن آبل من خلال إزالة المنفذ على جعل هاتفها الجديد "أرفع"، ومضاد للماء فإزالة فتحة السماعة، ستمنع تسرب الماء إلى الجهاز عبر الثقب، وتعطيله. ومن شأن إزالة المنفذ الذي يصل قطره إلى 2.5 ميلليمتر أن ينعكس إيجابا أيضاً على البطارية، على اعتبار أن التغيير سيفسح مساحة جديدة يمكن استغلالها. إلا أن التقرير لم يفصل كيف يمكن لمنفذ الشحن الجديد أن يمنع بدوره تسرب الماء إلى داخل الهاتف. في المقابل، يرى بعض منتقدي الشكل الجديد أو التغيير المنتظر أن الاستغناء عن السماعات التقليدية، سيجبر المستخدمين على شراء السماعات الأغلى التي تعمل بتقنية "بلوتوث".

Predicted Category: Tech

Example 7
Article content:
} قال رسول الله صلى الله عليه وسلم: «من أتى فراشه وهو ينوي أن يقوم يصلي من الليل فغلبته عينه حتى أصبح، كتب له ما نوى، وكان نومه صدقة عليه من ربه».} وقال صلى الله عليه وسلم: «ما تشاور قومإلا هداهم الله لأرشد أمورهم».

Predicted Category: Religion
'''

few_pred = []
for text in data['content']:
    prompt = f'''The article you need to classify
    Article:
    {text}
    Predicted Category:'''
    messages = [
        {"role": "system", "content": content},
        {"role": "user", "content": prompt}
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=512
    )
    generated_ids = [
        output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]

    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    few_pred.append(response)

In [ ]:
pred_few = pd.DataFrame()
pred_few['Predicted'] = few_pred
pred_few['Predicted'].value_counts()

,count
Predicted,
Predicted Category: Politics,134
Predicted Category: Medical,129
Culture,113
Sports,96
Predicted Category: Tech,75
...,...
"Predicted Category: Culture\n\nThis article primarily focuses on the work of a religious figure (the speaker, S卢女士的文章主要讨论了她最近为叙利亚难民所做的慈善旅行。她提到自己选择了作为传教士的道路，因为这是先知和改革者们所走的路，并通过加入一家名为社会改良协会的组织来实现这一目标。她在文章中详细描述了访问黎巴嫩、土耳其和约旦的难民营的经历，看到了难民们面临的困境，并参与了各种慈善项目，如提供食品援助、建立住房、医疗服务、教育项目等。她还提到了一些具体项目，如在也门进行的援助工作，以及在叙利亚支持的项目，如缝纫工坊、小额信贷项目和培训中心。最后，她强调这些组织的工作非常重要，因为政府提供的服务存在不足。\n\n基于以上内容，这篇文章的主要主题是关于文化和宗教活动中的慈善和社会工作。因此，最合适的分类应该是“文化”。\n",1
"Predicted Category: Religion\n\nThis article appears to be about the weekly religious publication ""al-Iman"" and includes instructions for readers to handle the paper respectfully, indicating its religious nature.",1
"Predicted Category: Science\n\nThe article discusses a scientific discovery related to an underground lake in Antarctica called Lake Vostok, which is of significant interest due to the potential for finding life forms that could provide clues about the possibility of life on other planets. The content focuses on scientific research and discoveries, making ""Science"" the most appropriate category among the given options. However, since ""Science"" is not one of the provided categories, the closest fit from the given options would be ""Medical,"" as some of the content touches on biological aspects, but it is primarily focused on the scientific exploration and implications of the discovery.",1


In [ ]:
nor_pre = []
for pr in pred_few['Predicted']:
  if (
      "الرياضة" in pr
      or "Sports" in pr
      or "sports" in pr
      ):
    nor_pre.append("Sports")
  elif (
      "الصحة" in pr
      or "الطب" in pr
      or "medicine" in pr
      or "Medicine" in pr
      or "Medical" in pr
      or "medical" in pr
      ):
    nor_pre.append("Medical")
  elif (
      "الثقافة" in pr
      or "Culture" in pr
      or "culture" in pr
      ):
    nor_pre.append("Culture")
  elif (
      "المال" in pr
      or "Finance" in pr
      or "finance" in pr
      ):
    nor_pre.append("Finance")
  elif (
      "السياسة" in pr
      or "politics" in pr
      or "Politics" in pr
      or "POLITICS" in pr
      ):
    nor_pre.append("Politics")
  elif (
      "الدين" in pr
      or "religion" in pr
      or "Religion" in pr
      or "religious" in pr
      or "Religious" in pr
      ):
    nor_pre.append("Religion")
  elif (
      "التكنولوجيا" in pr
      or "Tech" in pr
      or "tech" in pr
      or "Technology" in pr
      or "technology" in pr
      ):
    nor_pre.append("Tech")
  else:
    nor_pre.append("Unclassified")

pred_few['Normalized Category'] = nor_pre

In [ ]:
pred_few['Article'] = data['content']
pred_few.to_excel('Qwen-NC-FewShot-EnglishPrompt.xlsx', index = False)

In [ ]:
pred_few['Normalized Category'].value_counts()

,count
Normalized Category,
Politics,179
Culture,159
Medical,154
Sports,153
Finance,140
Tech,139
Religion,71
Other,5


In [ ]:
y_true = data['subdirectory'].values
print(classification_report(y_true, pred_few['Normalized Category'].values, digits = 4))

              precision    recall  f1-score   support

     Culture     0.8050    0.8533    0.8285       150
     Finance     0.8714    0.8133    0.8414       150
     Medical     0.9221    0.9467    0.9342       150
       Other     0.0000    0.0000    0.0000         0
    Politics     0.7877    0.9400    0.8571       150
    Religion     0.9718    0.6900    0.8070       100
      Sports     0.9477    0.9667    0.9571       150
        Tech     0.9281    0.8600    0.8927       150

    accuracy                         0.8760      1000
   macro avg     0.7792    0.7588    0.7648      1000
weighted avg     0.8865    0.8760    0.8774      1000



# CoT

In [ ]:
content = '''You are a language model tasked with classifying Arabic newspaper articles into one of the predefined editorial categories based solely on the article's main topic and content. Carefully read each article and assign only one of the following categories that best reflects its primary subject matter:
1. Culture
2. Finance
3. Medical
4. Politics
5. Religion
6. Sports
7. Tech

Follow these steps when analyzing the content of each article:

Step 1: Comprehend the Core Content
Read the full article carefully. Identify the central event, issue, or message. This may be a news story, commentary, or report involving domains like politics, health, economy, sports, technology, or culture.

Step 2: Identify the Dominant Theme
Determine the primary topic or theme. Focus on the article’s content and recurring concepts. Ignore secondary topics or tangents that do not define the article’s overall focus.

Step 3: Match the Article to a Category
Select the single category that best corresponds to the dominant theme. Be precise: choose the most specific and directly related category. Do not rely on assumptions or background knowledge outside the text. If the article touches on multiple themes, choose the one most emphasized or most central to the article’s purpose.


Example 1
Article content:
أكد الفنان عزت أبوعوف، رئيس مهرجان القاهرة السينمائي، إلغاء حفل ختام المهرجان الذي كان من المزمع أن يقام غداً، وذلك بسبب الظروف غير المستقرة التي تمر بها مصر حالياً.ونفى أبوعوف في تصريحات لـ"العربية.نت" سفر الضيوف الأجانب خوفاً مما تشهده مصر من مظاهرات، وما تردد عن أن هذا السفر قد تسبب في عدم إمكانية إقامة حفل الختام، مؤكداً أن وزير الثقافة دكتور محمد صابر عرب هو من أصدر قرار الإلغاء.وعن توزيع الجوائز التي كان من المزمع أن يتم في حفل الختام، شرح أنه سيقام مؤتمر صحافي سيكون بديلاً عن الحفل وسيتم من خلاله إعلان توزيع الجوائز والأفلام الفائزة في هذه الدورة.يُذكر أن حفل افتتاح المهرجان أيضاً تم في هدوء وبعيداً عن الصخب المعتاد، وذلك أيضاً بسبب الظروف نفسها التي تشهدها مصر.

Thoughts:
- Main content: cancellation of the closing ceremony of the Cairo International Film Festival
- Main theme: cultural and artistic events
- Category Milan: Culture

Predicted Category: 'Culture'

Example 2
Article content:
قال الرئيس التنفيذي للشركة السعودية للكهرباء زياد الشيحة، في مقابلة عبر الهاتف مع قناة "العربية"، إنه لأول مرة في تاريخ الشركة تراجع استهلاك المملكة في فترات الذروة. وأضاف الشيحة أن التراجع الذي حدث في استهلاك المملكة في 2016، مقارنة مع عام 2015، دفع الشركة لمراجعة السعات المطلوبة لدى دراسة المشاريع الجديدة. وأكد أن مسألة انخفاض الحمل الذروي لأول مرة في تاريخ الشركة عن العام جلعنا نراجع المحطات المستقبلية والتي ستكون بعقود شراء الطاقة، وهذا سيكون لمشاربع الإنتاج وتتم مراجعة السعات المطلوبة خاصة مع قلة الحمل الذروي في 2016. وستزود الشركة السعودية للكهرباء شركة زين السعودية بشبكة الألياف البصرية الممتدة لـ 60 ألف كم. وأوضح الشيحة أن "قطاع التوليد سيطرح للخصخصة كما هو معلن، ونعمل على الموضوع بشكل متوازن وشبه يومي". وكانت خسائر شركة السعودية للكهرباء قد تفاقمت بأكثر من 60%، في الربع الأخير من العام الماضي، مقارنة بالربع المماثل من عام 2015، لتبلغ 2.34 مليار ريال. من ناحية أخرى، ارتفعت أرباح الشركة بنسبة 37%، خلال العام الماضي، مقارنةً بعام 2015، لتبلغ 2.1 مليار ريال. وأرجعت الشركة تفاقم الخسائر الفصلية إلى ارتفاع تكلفة المبيعات نتيجة الزيادة في أسعار الوقود وارتفاع المصاريف التشغيلية.

Thoughts:
- Main content: interview with Ziyad Al-Shiha, CEO of the Saudi Electricity Company
- Main theme: Economy and energy
- Category Milan: Finance

Predicted category: 'Finance'

Example 3
Article content:
يعتقد بعض المدخنين أن السجائر الإلكترونية تعد أحد أهم العوامل المساعدة في الإقلاع عن التدخين، في حين يعتقد البعض الآخر أنها تعتبر وسيلة إغواء للاستمرار في الخضوع لتلك العادة المدمرة، إلا أن الأبحاث الطبية الحديثة تشير إلى أن الأخطار والفوائد لتلك النوعية من السجائر لاتزال غير معلومة بصورة واضحة بين المدخنين، وذلك وفق ما نشرت وكالة أنباء الشرق الأوسط المصرية. وكانت مجموعة من الباحثين قد أجرت أبحاثها على أكثر من 64 مدخنا، ولم ينجحوا في تحقيق إجماع حول الفوائد والأضرار المحتملة للسجائر الإلكترونية، وهو ما قد يعكس انقساما في المجتمع الطبي حول مدى ملاءمة تعزيز السجائر الإلكترونية كبديل أكثر أمنا للتدخين. وأوضح الباحثون أن معظم المشاركين في الدراسة يرون أن التدخين يعتبر شكلا من الإدمان، حيث تلعب الإرادة دورا قويا في الإقلاع عن هذه العادة المدمرة، في الوقت الذى حاول فيه جميع المشاركين في الدراسة مرة واحدة على الأقل الإقلاع عن العادة المدمرة.

Thoughts:
- Main Content: electronic cigarettes and their role in quitting smoking
- Main Theme: Health and medicine
- Category Match: Medical

Predicted Category: ' Medical'

Example 4
Article content:
أعرب المتحدث باسم الهيئة العليا للمفاوضات السورية سالم المسلط عن أمله في أن تنتقل روسيا فعليا لتقف إلى جانب الشعب السوري بدلا من النظام، وذلك عقب قراره بسحب القوات الروسية من سوريا. وأضاف المسلط أن هناك جدية لمست مؤخرا حيال المواقف الروسية للدفع نحو الحل السياسي للأزمة في سوريا، خلال جولة المحادثات الجديدة التي انطلقت في جنيف. هذا وأعلن متحدث باسم الرئيس الروسي فلاديمير بوتين بوتين بأن روسيا أبلغت الأسد بقرار سحب الجزء الرئيسي من القوات الروسية من سوريا. وقال المتحدث إن بوتين خلال اجتماعه بوزير دفاعه أمر اعتبارا من اليوم (الثلاثاء) ببدء سحب الجزء الرئيسي من القوات الروسية. في حين قال متحدث باسم بوتين إن القاعدة البحرية والجوية الروسية في سوريا تستمر في العمل كما في السابق. ووفقا للكرملين فإن بوتين طلب من وزير خارجيته سيرغي لافروف تكثيف الدور الروسي في عملية السلام في سوريا، مشيرا إلى ان  القوات الروسية في سوريا أوجدت ظروفا ملائمة لعملية السلام.

Thoughts:
- Main content: statements from Salem Al-Muslat, spokesperson for the Syrian High Negotiations Committee
- Main theme: Politics and international relations
- Category Milan: Politics

Predicted Category: ' Politics'

Example 5
Article content:
أكد عبداللطيف بخاري، رئيس لجنة الخبراء باتحاد القدم السعودي أن اتحاده يبحث عن 15 منصباً في اللجان الآسيوية، وأن هناك لجنة برئاسة خالد المرزوقي، عضو مجلس إدارة الاتحاد مكلفة بالاختيار. وقال بخاري لـ"في المرمى":" الترشيحات تتم بناء على معايير قارية، ونحن نبحث عن 15 منصباً في لجان الاتحاد الآسيوي". وبين بخاري أن لجنة المسابقات اقترحت إيجاد مراقب لكل مباراة، وزاد:" تقارير المراقب لن تغني عن تقارير الحكم، أما بالنسبة لتقارير الأول فيمكن للجنة الانضباط الاستناد عليها".

Thoughts:
- Main content: statements by Abdul Latif Bukhari, head of the Experts Committee at the Saudi Football Federation
- Main theme: Sports
- Category Milan: Sports

Predicted Category: ' Sports'

Example 6
Article content:
يبدو أن هاتف #آيفون7  الجديد الذي ستصدره شركة آبل، لن يحمل الكثير من التغيرات، بحسب ما أفاد تقرير لـ "وول ستريت جورنول". فالهاتف الجديد سيأتي شبيها بالنسخة الحالية (آيفون6)، مع تغيير جذري على صعيد "الصوت" والسماعات. وحسب تقرير الصحيفة قد تزيل شركة #آبل منفذ سماعة الصوت، لتدمجه بالمنفذ الذي يوضع فيه شاحن الهاتف في الأسفل. وتراهن آبل من خلال إزالة المنفذ على جعل هاتفها الجديد "أرفع"، ومضاد للماء فإزالة فتحة السماعة، ستمنع تسرب الماء إلى الجهاز عبر الثقب، وتعطيله. ومن شأن إزالة المنفذ الذي يصل قطره إلى 2.5 ميلليمتر أن ينعكس إيجابا أيضاً على البطارية، على اعتبار أن التغيير سيفسح مساحة جديدة يمكن استغلالها. إلا أن التقرير لم يفصل كيف يمكن لمنفذ الشحن الجديد أن يمنع بدوره تسرب الماء إلى داخل الهاتف. في المقابل، يرى بعض منتقدي الشكل الجديد أو التغيير المنتظر أن الاستغناء عن السماعات التقليدية، سيجبر المستخدمين على شراء السماعات الأغلى التي تعمل بتقنية "بلوتوث".

Thoughts:
- Main content: a report on Apple’s upcoming iPhone 7
- Main theme: Technology
- Category Milan: Technology

Predicted Category: ' Tech'

Example 7
Article content:
} قال رسول الله صلى الله عليه وسلم: «من أتى فراشه وهو ينوي أن يقوم يصلي من الليل فغلبته عينه حتى أصبح، كتب له ما نوى، وكان نومه صدقة عليه من ربه».} وقال صلى الله عليه وسلم: «ما تشاور قومإلا هداهم الله لأرشد أمورهم».

Thoughts:
- Main content: Hadiths (sayings of the Prophet Muhammad peace be upon him)
- Main theme: Religion and creed
- Category Milan: Religion

Predicted Category: ' Religion'
'''

cot_pred = []
for text in data['content']:
    prompt = f'''The article you need to classify
    Article:
    {text}'''
    messages = [
        {"role": "system", "content": content},
        {"role": "user", "content": prompt}
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=512
    )
    generated_ids = [
        output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]

    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    cot_pred.append(response)

In [ ]:
pred_cot = pd.DataFrame()
pred_cot['Predicted'] = cot_pred
pred_cot['Predicted'].value_counts()

,count
Predicted,
Thoughts:\n- Main content: Announcement of Microsoft's major update to Windows 10\n- Main theme: Technology and software updates\n- Category Milan: Technology\n\nPredicted Category: 'Tech',1
"Thoughts:\n- Main content: The election of Mohammed Saeed Al-Afeefi as the president of the International Theatre Institute (ITI) and the celebration of this achievement by the UAE's theatrical community.\n- Main theme: Recognition and achievements in the field of theater and cultural leadership.\n- Category Match: The article primarily focuses on the cultural and artistic achievements of the UAE, particularly in theater.\n\nPredicted Category: 'Culture'",1
Thoughts:\n- Main content: Exhibition of calligraphy and decorative arts organized by the Sharjah Calligraphy and Arts Center\n- Main theme: Art and culture\n- Category Match: Culture\n\nPredicted Category: 'Culture',1
"Thoughts:\n- Main content: Description of the Unlimited Love Institute established by Case Western Reserve University in Cleveland.\n- Main theme: Exploring the concept of unconditional love through scientific research and its impact on human life.\n- Category Match: The primary focus is on the scientific exploration and understanding of love, which falls under broader human behavior and values.\n\nPredicted Category: 'Culture'",1
"Thoughts:\n- Main content: Discussion and highlights from the Khaleej Foundation Award pavilion, including various lectures and workshops.\n- Main theme: Cultural and educational aspects, emphasizing the importance of education and excellence.\n- Category Match: The article focuses on cultural and educational achievements and initiatives, particularly related to the importance of education in fostering excellence and cultural development.\n\nPredicted Category: 'Culture'",1
...,...
"Thoughts:\n- Main content: A lecture on Napoleon Bonaparte's campaigns in Egypt during 1797-1799 at Sorbonne University Abu Dhabi.\n- Main theme: Historical analysis and discussion focusing on historical events and figures.\n- Category Match: The primary focus is on historical events and figures rather than purely cultural, political, medical, financial, religious, or technological topics.\n\nPredicted Category: 'Culture'",1
Thoughts:\n- Main content: The artist Lita Cabellut is organizing an art exhibition in Dubai next year.\n- Main theme: Art and culture.\n- Category Match: Culture\n\nPredicted Category: 'Culture',1
"Thoughts:\n- Main content: Results and details of a poetry competition show called ""Shayar al Million"" (Shayar al Million is the Arabic name for ""Poet of the Million"")\n- Main theme: Cultural event and poetry\n- Category Milan: Culture\n\nPredicted Category: 'Culture'",1


In [ ]:
nor_pre = []
for pr in pred_cot['Predicted']:
  if (
      "الرياضة" in pr
      or "Sports" in pr
      or "sports" in pr
      ):
    nor_pre.append("Sports")
  elif (
      "الصحة" in pr
      or "الطب" in pr
      or "medicine" in pr
      or "Medicine" in pr
      or "Medical" in pr
      or "medical" in pr
      ):
    nor_pre.append("Medical")
  elif (
      "الثقافة" in pr
      or "Culture" in pr
      or "culture" in pr
      ):
    nor_pre.append("Culture")
  elif (
      "المال" in pr
      or "Finance" in pr
      or "finance" in pr
      ):
    nor_pre.append("Finance")
  elif (
      "السياسة" in pr
      or "politics" in pr
      or "Politics" in pr
      or "POLITICS" in pr
      ):
    nor_pre.append("Politics")
  elif (
      "الدين" in pr
      or "religion" in pr
      or "Religion" in pr
      or "religious" in pr
      or "Religious" in pr
      ):
    nor_pre.append("Religion")
  elif (
      "التكنولوجيا" in pr
      or "Tech" in pr
      or "tech" in pr
      or "Technology" in pr
      or "technology" in pr
      ):
    nor_pre.append("Tech")
  else:
    nor_pre.append("Unclassified")

pred_cot['Normalized Category'] = nor_pre

In [ ]:
pred_cot['Article'] = data['content']
pred_cot.to_excel('Qwen-NC-CoT-EnglishPrompt.xlsx', index = False)

In [ ]:
pred_cot['Normalized Category'].value_counts()

,count
Normalized Category,
Politics,181
Medical,155
Sports,153
Finance,147
Culture,145
Tech,130
Religion,87
Economy,1
Unclassified,1


In [ ]:
y_true = data['subdirectory'].values

In [ ]:
print(classification_report(y_true, pred_cot['Normalized Category'].values, digits = 4))

              precision    recall  f1-score   support

     Culture     0.8483    0.8200    0.8339       150
     Economy     0.0000    0.0000    0.0000         0
     Finance     0.8367    0.8200    0.8283       150
     Medical     0.9226    0.9533    0.9377       150
    Politics     0.7680    0.9267    0.8399       150
    Religion     0.9310    0.8100    0.8663       100
      Sports     0.9346    0.9533    0.9439       150
        Tech     0.9692    0.8400    0.9000       150
Unclassified     0.0000    0.0000    0.0000         0

    accuracy                         0.8780      1000
   macro avg     0.6901    0.6804    0.6833      1000
weighted avg     0.8850    0.8780    0.8792      1000

